# Title: Hospital Bill Prediction Using Machine Learning (using Decision Trees and Random Forest)

Objective: To predict hospital bill amounts using patient and treatment-related information and evaluate the performance of Decision Tree and Random Forest regression models through 5-Fold Cross Validation for accurate healthcare cost estimation.

# Importing Libraries

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [3]:
from sklearn.model_selection import KFold

In [4]:
from sklearn.preprocessing import OneHotEncoder

In [5]:
from sklearn.impute import SimpleImputer

In [6]:
url = "https://raw.githubusercontent.com/YBIFoundation/ProjectDataSet/main/Hospital%20Bill%20Prediction%20for%20MediCarePlus.csv"

df = pd.read_csv(url)

# Understanding the Dataset

In [7]:
df.head()

,gender,age,insurance_type,diagnosis_type,procedures_count,surgery_done,icu_admission,length_of_stay_days,medications_count,room_type,predicted_hospital_bill_inr
0,Male,79,Government,Maternity,3,Yes,No,5,6,General Ward,41254.0
1,Other,70,Private,Cardiology,3,No,No,11,1,General Ward,37620.0
2,Male,0,Government,Maternity,3,Yes,Yes,8,4,Private,64684.0
3,Male,11,Government,Maternity,5,No,No,17,12,General Ward,61271.0
4,Male,7,NaN,General,1,Yes,Yes,6,5,General Ward,65793.0


In [13]:
df.shape

(1000, 11)

In [10]:
df.describe()

,age,procedures_count,length_of_stay_days,medications_count,predicted_hospital_bill_inr
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.0000
mean,46.581000,2.010000,14.631000,4.910000,58730.7640
std,26.112512,1.459462,8.395912,2.316304,22004.0488
min,0.000000,0.000000,1.000000,0.000000,9770.0000
25%,23.000000,1.000000,7.000000,3.000000,41480.2500
50%,47.000000,2.000000,14.000000,5.000000,58423.0000
75%,70.000000,3.000000,22.000000,6.000000,73666.7500
max,89.000000,8.000000,29.000000,14.000000,134779.0000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gender                       1000 non-null   object 
 1   age                          1000 non-null   int64  
 2   insurance_type               801 non-null    object 
 3   diagnosis_type               1000 non-null   object 
 4   procedures_count             1000 non-null   int64  
 5   surgery_done                 1000 non-null   object 
 6   icu_admission                1000 non-null   object 
 7   length_of_stay_days          1000 non-null   int64  
 8   medications_count            1000 non-null   int64  
 9   room_type                    1000 non-null   object 
 10  predicted_hospital_bill_inr  1000 non-null   float64
dtypes: float64(1), int64(4), object(6)
memory usage: 86.1+ KB


From the above cell we can see that there are missing calues in the insurance_type column

In [12]:
df.columns

Index(['gender', 'age', 'insurance_type', 'diagnosis_type', 'procedures_count',
       'surgery_done', 'icu_admission', 'length_of_stay_days',
       'medications_count', 'room_type', 'predicted_hospital_bill_inr'],
      dtype='object')

In [14]:
#Checking missing values
df.isnull().sum()

,0
gender,0
age,0
insurance_type,199
diagnosis_type,0
procedures_count,0
surgery_done,0
icu_admission,0
length_of_stay_days,0
medications_count,0
room_type,0


# Separating the X and y datasets

In [15]:
y = df['predicted_hospital_bill_inr']

In [16]:
X = df.drop(
    columns=['predicted_hospital_bill_inr']
)

In [17]:
X.head(10)

,gender,age,insurance_type,diagnosis_type,procedures_count,surgery_done,icu_admission,length_of_stay_days,medications_count,room_type
0,Male,79,Government,Maternity,3,Yes,No,5,6,General Ward
1,Other,70,Private,Cardiology,3,No,No,11,1,General Ward
2,Male,0,Government,Maternity,3,Yes,Yes,8,4,Private
3,Male,11,Government,Maternity,5,No,No,17,12,General Ward
4,Male,7,NaN,General,1,Yes,Yes,6,5,General Ward
5,Male,23,Government,Maternity,4,No,No,16,6,Semi-Private
6,Other,41,NaN,General,2,Yes,Yes,27,5,Semi-Private
7,Male,0,Government,Cardiology,3,No,Yes,17,10,Private
8,Female,55,Government,Oncology,2,No,No,5,9,Semi-Private
9,Other,16,Private,General,3,No,No,28,1,Private


#Identifying Numerical and Categorical Columns

In [18]:
num_cols = X.select_dtypes(
    include=['int64','float64']
).columns

In [19]:
cat_cols = X.select_dtypes(
    include=['object']
).columns

In [20]:
print(num_cols)
print(cat_cols)

Index(['age', 'procedures_count', 'length_of_stay_days', 'medications_count'], dtype='object')
Index(['gender', 'insurance_type', 'diagnosis_type', 'surgery_done',
       'icu_admission', 'room_type'],
      dtype='object')


# Handling Missing Values

In [21]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

In [22]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='Unknown'
    )),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

# Combining Both the pipelines

In [23]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

Since the dataset is very small (only 1000 rows), I am thinking of using Cross Validation Strategy instead of train_test_split

In [24]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

#Decision Tree Model Pipeline

In [25]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(
        random_state=42
    ))
])

In [26]:
dt_scores = cross_val_score(
    dt_pipeline,
    X,
    y,
    cv=kf,
    scoring='r2'
)

print("R² Scores:")
print(dt_scores)

print("\nAverage R²:")
print(dt_scores.mean())

R² Scores:
[0.83664676 0.81422704 0.82618892 0.82700142 0.84550311]

Average R²:
0.8299134494817988


# Random Forest Pipeline

In [27]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [28]:
rf_scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=kf,
    scoring='r2'
)

print("R² Scores:")
print(rf_scores)

print("\nAverage R²:")
print(rf_scores.mean())

R² Scores:
[0.91379667 0.91693121 0.920666   0.89107694 0.91925353]

Average R²:
0.9123448711741547


In [29]:
print("Decision Tree Average R²:",
      dt_scores.mean())

print("Random Forest Average R²:",
      rf_scores.mean())

Decision Tree Average R²: 0.8299134494817988
Random Forest Average R²: 0.9123448711741547


# Checking variability of the models

In [30]:
print("Decision Tree Std:",
      dt_scores.std())

print("Random Forest Std:",
      rf_scores.std())

Decision Tree Std: 0.010552632914551148
Random Forest Std: 0.01088519844229524
